## Configuring Thinking in Claude Messages API

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")


### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = anthropic_api_key)

### Calling Messages API with Thinking Param

In [ ]:
user_prompt = """
You're a career coach having expertize in GenAI. A job seeker has the following skill gaps:

- LLM integration
- On-device AI
- Model optimization
- GenAI patterns

Recommend a learning path for the job seeker to fill these skill gaps. Provide a list of resources, courses, and projects that can help them gain expertise in these areas.
""" 

response = client.messages.create(
    model = claude_model,
    max_tokens = 1024,
    # thinking = "medium",
    thinking = {"type": "adaptive", "display": "summarized"},
    messages = [
        {"role": "user", "content": user_prompt}
    ]
)

for block in response.content:
    if block.type == "thinking":
        print(f"\nThinking: {block.thinking}")
    elif block.type == "text":
        print(f"\nResponse: {block.text}")

### Calling Messages API with Thinking Param, then Streaming response

In [ ]:
with client.messages.stream(
    model = claude_model,
    max_tokens = 1024,
    # thinking = "medium",
    thinking = {"type": "adaptive", "display": "summarized"},
    messages = [
        {"role": "user", "content": user_prompt}
    ]
) as stream:
    for event in stream:
        if event.type == "content_block_start":
            print(f"\nStarting {event.content_block.type} block...")
        elif event.type == "content_block_delta":
            if event.delta.type == "thinking_delta":
                print(event.delta.thinking, end = "", flush = True)
            elif event.delta.type == "text_delta":
                print(event.delta.text, end = "", flush = True)